In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [ ]:
df = pd.read_csv("data/dataset_copy.csv").drop(["track_num"],axis=1).dropna(subset = ['track_name'])
df.head()

,track_id,artist_name,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson & ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
df["artist_track_name"] = df["artist_name"] + " - "+ df["track_name"]
df

,track_id,artist_name,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre,artist_track_name
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,...,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic,Gen Hoshino - Comedy
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,...,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic,Ben Woodward - Ghost - Acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson & ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,...,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic,Ingrid Michaelson & ZAYN - To Begin Again
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,...,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic,Kina Grannis - Can't Help Falling In Love
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,...,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic,Chord Overstreet - Hold On
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,5,...,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music,Rainy Lullaby - Sleep My Little Boy
113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,0,...,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music,Rainy Lullaby - Water Into Light
113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,0,...,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music,Cesária Evora - Miss Perfumado
113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,7,...,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music,Michael W. Smith - Friends


In [ ]:
df.drop_duplicates(subset=['artist_track_name'], keep=False)
df=df.groupby('genre').sample(frac=.2).reset_index(drop=True)
df


,track_id,artist_name,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre,artist_track_name
0,2PTzfXh5Ut3hjRrWGyafz6,Stephen Speaks,No More Doubt,Passenger Seat,64,274480,False,0.523,0.554,7,...,1,0.0303,0.7490,0.000003,0.3230,0.7690,152.979,4,acoustic,Stephen Speaks - Passenger Seat
1,0v0pc1lIt5p6EBX7pnfOGF,Ross Copperman,Holding On and Letting Go LP,Holding On and Letting Go,49,317441,False,0.486,0.555,2,...,1,0.0271,0.8180,0.002690,0.1090,0.0992,77.987,4,acoustic,Ross Copperman - Holding On and Letting Go
2,4qTHoIp62ngNDpUJwW3hZ7,Boyce Avenue,"Cover Sessions, Vol. 6",Can’t Help Falling in Love,57,131000,False,0.402,0.197,5,...,1,0.0295,0.8580,0.000001,0.1140,0.1480,100.044,3,acoustic,Boyce Avenue - Can’t Help Falling in Love
3,33JX2be3eKVhl5xk8YQhVc,Ray LaMontagne,Till The Sun Turns Black,Empty,48,317280,False,0.482,0.381,2,...,1,0.0311,0.4550,0.002950,0.1140,0.4200,81.712,4,acoustic,Ray LaMontagne - Empty
4,6OCsvPU6P84wJ0erggCRv4,Albert King,pov: you have a holly jolly christmas,Christmas Comes But Once A Year,0,272640,False,0.687,0.494,1,...,1,0.0505,0.3240,0.000055,0.0867,0.6210,93.269,1,acoustic,Albert King - Christmas Comes But Once A Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22795,4WbOUe6T0sozC7z5ZJgiAA,Lucas Cervetti,Frecuencias Álmicas en 432hz,"Frecuencia Álmica, Pt. 4",22,305454,False,0.331,0.171,1,...,1,0.0350,0.9200,0.022900,0.0679,0.3270,132.147,3,world-music,"Lucas Cervetti - Frecuencia Álmica, Pt. 4"
22796,5QT13TE0vN29dYa8Ifkz0A,Leeland & TAYA,Heart & Flesh,Heart & Flesh,55,431520,False,0.270,0.386,2,...,1,0.0301,0.2690,0.000001,0.0995,0.1510,131.876,3,world-music,Leeland & TAYA - Heart & Flesh
22797,4PYOh644yKjYyvyVUHqcof,All Sons & Daughters & Leslie Jordan & David L...,Live,Great Are You Lord - Live,60,290413,False,0.390,0.457,9,...,1,0.0327,0.0874,0.000009,0.1230,0.1450,143.977,3,world-music,All Sons & Daughters & Leslie Jordan & David L...
22798,6hzaKM08IZOAWMoITG9UjV,Raul Paz,Ven Ven,No Me Digas Que No,25,207866,False,0.773,0.634,5,...,0,0.0754,0.3810,0.000000,0.1630,0.5650,135.991,4,world-music,Raul Paz - No Me Digas Que No


In [ ]:
pickle.dump(df, open('pickle/df_smaller.pkl','wb'))

In [ ]:
# since genre is in the form of different unique genres
mylist=df['artist_track_name'].to_list()
mylist = list(dict.fromkeys(mylist))
pickle.dump(mylist, open('pickle/mylist_smaller.pkl','wb'))
mylist

['Stephen Speaks - Passenger Seat',
 'Ross Copperman - Holding On and Letting Go',
 'Boyce Avenue - Can’t Help Falling in Love',
 'Ray LaMontagne - Empty',
 'Albert King - Christmas Comes But Once A Year',
 'Ben Rector - Brand New',
 'Amber Leigh Irish - Save Your Tears - Acoustic',
 'Zack Tabudlo - Habang Buhay',
 'Ramshackle Glory - Of Ballots and Barricades - Live',
 'Andrew Belle - Pieces',
 "Gabrielle Aplin - Please Don't Say You Love Me",
 'Frank Turner - Little Life',
 "JJ Heller - It'll Be Alright",
 'Kina Grannis - Iris',
 'Ingrid Michaelson - Everybody',
 'The Mayries - Overpass Graffiti',
 'The Bridge City Sinners - Laugh While You Can',
 'Boyce Avenue & Tyler Ward - Fix You',
 'Keisha White - The Weakness In Me',
 "Mother's Daughter & Beck Pete - A Sky Full of Stars",
 'Tyler Ward & Karis & Ray Lorraine - Love Story',
 'Jason Mraz & Meghan Trainor - More Than Friends (feat. Meghan Trainor)',
 'Anna Hamilton - Bad Liar',
 'Parachute - The Mess I Made',
 'Aoi Teshima - こころをこめ

In [ ]:
# import json
# config_filename = 'config.json'
# # Writing the dictionary to a file in JSON format
# with open(config_filename, 'w') as config_file:
#     json.dump(mylist, config_file)

# print(f"Data successfully written to {config_filename}")
# pickle.dump(mylist, open('mylist.pkl','wb'))

In [ ]:
# use OneHotEncoding to make each genre into booleans 
genre_encoded = pd.get_dummies(df["genre"])

In [ ]:
features = df[[
    'danceability', 'energy', 'loudness', 'instrumentalness', 'tempo', 
    'key', 'mode', 'popularity'
    # ,'duration_ms'
]]

X = pd.concat([features, genre_encoded], axis=1)
df['genre'].unique()

array(['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient',
       'anime', 'black-metal', 'bluegrass', 'blues', 'brazil',
       'breakbeat', 'british', 'cantopop', 'chicago-house', 'children',
       'chill', 'classical', 'club', 'comedy', 'country', 'dance',
       'dancehall', 'death-metal', 'deep-house', 'detroit-techno',
       'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm',
       'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk',
       'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove',
       'grunge', 'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle',
       'heavy-metal', 'hip-hop', 'honky-tonk', 'house', 'idm', 'indian',
       'indie', 'indie-pop', 'industrial', 'iranian', 'j-dance', 'j-idol',
       'j-pop', 'j-rock', 'jazz', 'k-pop', 'kids', 'latin', 'latino',
       'malay', 'mandopop', 'metal', 'metalcore', 'minimal-techno', 'mpb',
       'new-age', 'opera', 'pagode', 'party', 'piano', 'pop', 'pop-film',
       'pow

In [ ]:
# drop rows with missing values (track_name)
# df = df.dropna(subset = ['track_name'])

In [ ]:
# make data into scaled bc knn depends on the distance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pickle.dump(X_scaled, open('pickle/X_scaled_smaller.pkl','wb'))


In [ ]:
# use euclidean distance

knn = NearestNeighbors(
    n_neighbors=10,
    metric="euclidean"
)

knn.fit(X_scaled)
pickle.dump(knn, open('pickle/knn_smaller.pkl','wb'))

In [ ]:
# lookup track by track_name
def get_song_index(song_name):
    return df[df["artist_track_name"] == song_name].index[0]

In [ ]:
# get la neighbors for that song
def get_neighbors(song_name):
    index = get_song_index(song_name)
    print(index)
    song_vector = X_scaled[index].reshape(1,-1)
    distances, indices = knn.kneighbors(song_vector)

    return distances[0], indices[0]

In [ ]:
def genre_bonus(index, song_A, song_B):
    genre_A = df[df["artist_track_name"] == song_A]["genre"].values[0]
    genre_B = df[df["artist_track_name"] == song_B]["genre"].values[0]

    genre_C = df.iloc[index]["genre"]

    bonus = 0

    if genre_C == genre_A:
        bonus += 0.1
    if genre_C == genre_B:
        bonus += 0.1

    return bonus

In [ ]:
def bridge_recommendation(song_A, song_B, top_n=5): 
    dist_A, ind_A = get_neighbors(song_A) # the distance and the independent variables chosen
    dist_B, ind_B = get_neighbors(song_B)

    scores = {}

    # convert those distances into similarity
    for d, i in zip(dist_A, ind_A):
        # scores[i] = scores.get(i, 0) + (1 / (1 + d)) # without the added weight of genre(OneHotEncoding)
        scores[i] = scores.get(i, 0) + (1 / (1 + d)) + genre_bonus(i, song_A, song_B)

    for d, i in zip(dist_B, ind_B):
        # scores[i] = scores.get(i, 0) + (1 / (1 + d))
        scores[i] = scores.get(i, 0) + (1 / (1 + d)) + genre_bonus(i, song_A, song_B)

    # sort by combined score
    sorted_songs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for i, score in sorted_songs:
        name= df.iloc[i]["artist_track_name"]

        if name not in [song_A, song_B]:
            results.append((name, score))

    return results[:top_n]

In [ ]:
def normalize_scores(results):
    if not results:
        return []
    
    max_score = max(score for _, score in results)

    output = []
    for name, score in results:
        confidence = score / max_score
        output.append((name, score, confidence))

    return output

In [ ]:
song_1=df.sample(n=1)["artist_track_name"].values[0]
song_2=df.sample(n=1)["artist_track_name"].values[0]

print(song_1)
print(song_2)


Deluxe - Blue
Baron Vaughn - Baron Vaughn


In [ ]:
results = bridge_recommendation(song_1,song_2,4)
results


20987
3603


[('Larry The Cable Guy - Skeeters, Spiders and Lookalikes',
  np.float64(0.6840504657287757)),
 ('Rhod Gilbert - Escape to Afghanistan', np.float64(0.6140402554398546)),
 ('George Lopez - Chicano Dudes', np.float64(0.6026455859995494)),
 ('Stuart Thompson - Gun Range', np.float64(0.5996140511461078))]

In [ ]:
# import pickle
# model = pickle.load(open('model.pkl','rb'))
# model

In [ ]:
normalized_results = normalize_scores(results)
normalized_results

[('Larry The Cable Guy - Skeeters, Spiders and Lookalikes',
  np.float64(0.6840504657287757),
  np.float64(1.0)),
 ('Rhod Gilbert - Escape to Afghanistan',
  np.float64(0.6140402554398546),
  np.float64(0.8976534425507139)),
 ('George Lopez - Chicano Dudes',
  np.float64(0.6026455859995494),
  np.float64(0.8809957981060668)),
 ('Stuart Thompson - Gun Range',
  np.float64(0.5996140511461078),
  np.float64(0.8765640565821254))]

In [ ]:
# score is the raw value and confidence is the normalized value (0-1)
for name, score, confidence in normalized_results:
    print(f"{name} → Score: {score:.4f} | Confidence: {confidence:.2%}")

Larry The Cable Guy - Skeeters, Spiders and Lookalikes → Score: 0.6841 | Confidence: 100.00%
Rhod Gilbert - Escape to Afghanistan → Score: 0.6140 | Confidence: 89.77%
George Lopez - Chicano Dudes → Score: 0.6026 | Confidence: 88.10%
Stuart Thompson - Gun Range → Score: 0.5996 | Confidence: 87.66%
